In [7]:
import hashlib
from PIL import Image
from pathlib import Path
from typing import List

class ImageScanner:
    """
    Lớp này chịu trách nhiệm quét thư mục gốc (hỗ trợ đọc sâu vào các thư mục con), 
    lọc các định dạng ảnh hợp lệ và loại bỏ ảnh trùng lặp bằng mã băm MD5.
    """
    
    def __init__(self, dataset_dir: str):
        """
        Khởi tạo đối tượng ImageScanner.
        
        Input:
            dataset_dir (str): Đường dẫn đến thư mục chứa ảnh thô trên Kaggle Input.
        """
        self.dataset_dir = Path(dataset_dir)
        # Bổ sung thêm .webp, .jpeg vì ảnh tải từ Pinterest thường dính định dạng này
        self.valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}

    def _generate_md5(self, file_path: Path) -> str:
        """
        Tạo mã băm MD5 cho một tệp tin để nhận diện nội dung.
        
        Input:
            file_path (Path): Đường dẫn đến tệp tin cần băm.
            
        Output:
            str: Chuỗi mã băm MD5 của tệp tin.
        """
        hash_md5 = hashlib.md5()
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()

    def get_unique_filepaths(self) -> List[Path]:
        """
        Thực thi quá trình quét đệ quy và lọc ảnh trùng lặp.
        
        Output:
            List[Path]: Danh sách các đường dẫn tuyệt đối trỏ đến những bức ảnh duy nhất (không trùng lặp).
        """
        print(f"Scanning for images in directory: {self.dataset_dir}")
        seen_hashes = set()
        unique_paths = []
        duplicate_count = 0

        # Sử dụng rglob("*") để quét đệ quy tất cả các thư mục con (nếu có)
        for file_path in self.dataset_dir.rglob("*"):
            if file_path.is_file() and file_path.suffix.lower() in self.valid_extensions:
                file_hash = self._generate_md5(file_path)
                
                if file_hash in seen_hashes:
                    duplicate_count += 1
                else:
                    seen_hashes.add(file_hash)
                    unique_paths.append(file_path)
                    
        print(f"Scan completed. Found {len(unique_paths)} unique images. Ignored {duplicate_count} duplicates.")
        return unique_paths


class ImageFormatter:
    """
    Lớp này chịu trách nhiệm chuẩn hoá hình ảnh từ danh sách đường dẫn đầu vào:
    chuyển đổi hệ màu sang RGB, thay đổi kích thước và cắt vùng trung tâm (Center Crop)
    sau đó lưu vào thư mục đầu ra trên Kaggle Working.
    """
    
    def __init__(self, output_dir: str, target_size: int = 512):
        """
        Khởi tạo đối tượng ImageFormatter.
        
        Input:
            output_dir (str): Đường dẫn thư mục cho phép ghi để lưu ảnh đã chuẩn hoá.
            target_size (int): Kích thước cạnh của ảnh vuông đầu ra (mặc định 512).
        """
        self.output_dir = Path(output_dir)
        self.target_size = target_size
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _center_crop_and_resize(self, image: Image.Image) -> Image.Image:
        """
        Cắt ảnh theo hình vuông ở giữa và thu phóng về kích thước mục tiêu.
        
        Input:
            image (Image.Image): Đối tượng ảnh PIL gốc.
            
        Output:
            Image.Image: Đối tượng ảnh PIL đã qua xử lý.
        """
        width, height = image.size
        min_dim = min(width, height)
        
        left = (width - min_dim) / 2
        top = (height - min_dim) / 2
        right = (width + min_dim) / 2
        bottom = (height + min_dim) / 2
        
        cropped_image = image.crop((left, top, right, bottom))
        resized_image = cropped_image.resize(
            (self.target_size, self.target_size), 
            Image.Resampling.LANCZOS
        )
        return resized_image

    def process(self, image_paths: List[Path]) -> None:
        """
        Thực thi quá trình chuẩn hoá cho danh sách đường dẫn ảnh.
        
        Input:
            image_paths (List[Path]): Danh sách các đường dẫn ảnh đã được lọc trùng.
        """
        print(f"Starting image formatting. Target size: {self.target_size}x{self.target_size}")
        processed_count = 0
        
        for file_path in image_paths:
            try:
                with Image.open(file_path) as img:
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                        
                    formatted_img = self._center_crop_and_resize(img)
                    
                    save_path = self.output_dir / f"{processed_count:04d}.jpg"
                    formatted_img.save(save_path, "JPEG", quality=95)
                    processed_count += 1
            except Exception as e:
                print(f"Error processing file {file_path.name}: {e}")
                
        print(f"Formatting completed. Successfully processed {processed_count} images in {self.output_dir}.")

In [8]:
import torch
from PIL import Image
from pathlib import Path
from transformers import BlipProcessor, BlipForConditionalGeneration

class BLIPCaptioner:
    """
    Lớp này chịu trách nhiệm duy nhất là sử dụng mô hình trí tuệ nhân tạo (BLIP) 
    để tự động phân tích hình ảnh, sinh ra các đoạn chú thích (caption) tiếng Anh 
    và lưu thành tệp văn bản (.txt) tương ứng phục vụ huấn luyện LoRA.
    """
    
    def __init__(self, dataset_dir: str, trigger_word: str = "nhatbinh dress, "):
        """
        Khởi tạo đối tượng BLIPCaptioner, tải mô hình từ HuggingFace và cấu hình thiết bị xử lý.
        
        Input:
            dataset_dir (str): Đường dẫn đến thư mục chứa ảnh đã được chuẩn hoá.
            trigger_word (str): Từ khoá kích hoạt để gắn vào đầu mỗi caption, giúp mô hình Stable Diffusion học khái niệm mới.
            
        Output:
            Không có (Khởi tạo đối tượng).
        """
        self.dataset_dir = Path(dataset_dir)
        self.trigger_word = trigger_word
        
        # Kiểm tra xem có GPU (CUDA) hay không để đẩy mô hình lên GPU, tối ưu hoá tốc độ tính toán
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading BLIP model on device: {self.device}")
        
        # Sử dụng phiên bản BLIP base vì nó cân bằng hoàn hảo giữa tốc độ và độ chính xác cho bài toán này
        self.processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(self.device)

    def _generate_caption(self, image_path: Path) -> str:
        """
        Đọc một bức ảnh cụ thể và sử dụng mô hình BLIP để sinh ra chuỗi văn bản miêu tả ảnh đó.
        
        Input:
            image_path (Path): Đường dẫn đến tệp tin ảnh cần xử lý.
            
        Output:
            str: Chuỗi văn bản miêu tả nội dung bức ảnh (đã được làm sạch các ký tự đặc biệt của hệ thống).
        """
        # Đảm bảo ảnh luôn ở định dạng RGB trước khi đưa vào mô hình AI
        raw_image = Image.open(image_path).convert('RGB')
        
        # Tiền xử lý ảnh thành tensor và đưa vào thiết bị tính toán (GPU/CPU)
        inputs = self.processor(raw_image, return_tensors="pt").to(self.device)
        
        # Sinh văn bản mô tả (giới hạn độ dài tối đa 50 token để tránh caption quá rườm rà)
        out = self.model.generate(**inputs, max_new_tokens=50)
        caption = self.processor.decode(out[0], skip_special_tokens=True)
        
        return caption

    def process(self) -> None:
        """
        Thực thi quá trình duyệt qua toàn bộ ảnh trong thư mục, sinh caption, 
        ghép thêm trigger_word và lưu ra tệp .txt có cùng tên với ảnh gốc.
        
        Input:
            Không có.
            
        Output:
            Không có (Kết quả là các tệp .txt được tạo ra trên ổ cứng).
        """
        print(f"Starting auto-captioning process in: {self.dataset_dir}")
        valid_extensions = {".jpg", ".jpeg", ".png"}
        processed_count = 0
        
        # Duyệt qua tất cả các file trong thư mục dữ liệu
        for file_path in self.dataset_dir.iterdir():
            if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
                try:
                    # Sinh caption thuần túy từ hình ảnh
                    raw_caption = self._generate_caption(file_path)
                    
                    # Ghép trigger word vào đầu câu. 
                    # Ví dụ: "nhatbinh dress, a woman standing next to a tree"
                    final_caption = f"{self.trigger_word}{raw_caption}"
                    
                    # Tạo đường dẫn lưu file .txt (giữ nguyên tên ảnh, chỉ đổi phần mở rộng)
                    txt_path = file_path.with_suffix(".txt")
                    
                    # Ghi caption đã hoàn thiện vào tệp
                    with open(txt_path, "w", encoding="utf-8") as f:
                        f.write(final_caption)
                        
                    processed_count += 1
                    
                    # Cập nhật tiến độ log để kiểm soát trên giao diện Kaggle
                    if processed_count % 50 == 0:
                        print(f"Processed {processed_count} images and generated captions...")
                        
                except Exception as e:
                    print(f"Error processing file {file_path.name}: {e}")
                    
        print(f"Captioning completed. Successfully generated {processed_count} text files.")

In [10]:
import re
from pathlib import Path

class CaptionSanitizer:
    """
    Lớp này chịu trách nhiệm duy nhất là làm sạch và chuẩn hoá văn bản.
    Nó tìm kiếm và loại bỏ các từ khoá ảo giác (hallucination) liên quan đến 
    các nền văn hoá khác mà mô hình VLM (BLIP) nhận diện sai, bảo vệ tính 
    thuần khiết của tập dữ liệu huấn luyện LoRA.
    """
    
    def __init__(self, dataset_dir: str):
        """
        Khởi tạo đối tượng CaptionSanitizer.
        
        Input:
            dataset_dir (str): Đường dẫn đến thư mục chứa các tệp .txt đã được sinh ra bởi BLIP.
        """
        self.dataset_dir = Path(dataset_dir)
        
        # Danh sách các từ khoá (Regex) mà AI thường ảo giác
        # \b giúp khớp chính xác ranh giới của từ, tránh việc xoá nhầm các từ chứa chuỗi tương tự
        self.hallucinated_patterns = [
            r"\bkimonos?\b", r"\bhanboks?\b", r"\bhanfu\b", r"\bqipao\b", r"\byukata\b",
            r"\bchinese\b", r"\bjapanese\b", r"\bkorean\b", r"\basian\b",
            r"\bcostumes?\b" 
        ]
        
        # Cụm từ thay thế an toàn và trung tính
        self.replacement = "traditional attire"

    def _sanitize_text(self, text: str) -> str:
        """
        Thực thi biểu thức chính quy để tẩy rửa một chuỗi văn bản.
        
        Input:
            text (str): Chuỗi văn bản gốc do AI sinh ra.
            
        Output:
            str: Chuỗi văn bản đã được làm sạch và xoá khoảng trắng thừa.
        """
        cleaned_text = text.lower()
        
        # Quét và thay thế từng pattern sai lệch
        for pattern in self.hallucinated_patterns:
            cleaned_text = re.sub(pattern, self.replacement, cleaned_text)
            
        # Dọn dẹp các khoảng trắng thừa hoặc dấu phẩy bị lặp do quá trình thay thế
        cleaned_text = re.sub(r",\s*,", ",", cleaned_text)
        cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
        
        return cleaned_text

    def process(self) -> None:
        """
        Duyệt qua toàn bộ các tệp .txt trong thư mục và ghi đè nội dung đã được làm sạch.
        """
        print(f"Starting caption sanitization in directory: {self.dataset_dir}")
        sanitized_count = 0
        
        for file_path in self.dataset_dir.iterdir():
            if file_path.is_file() and file_path.suffix == ".txt":
                try:
                    # Đọc nội dung cũ
                    with open(file_path, "r", encoding="utf-8") as f:
                        original_text = f.read()
                        
                    # Làm sạch nội dung
                    clean_text = self._sanitize_text(original_text)
                    
                    # Ghi đè lại file
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(clean_text)
                        
                    sanitized_count += 1
                except Exception as e:
                    print(f"Error sanitizing file {file_path.name}: {e}")
                    
        print(f"Sanitization completed. Cleaned {sanitized_count} caption files.")

In [11]:
RAW_DIR = "/kaggle/input/datasets/thedestoroyah/ao-nhat-binh-v2/ao_nhat_binh" 
CLEAN_DIR = "/kaggle/working/dataset_formatted"

# 1. Quét tìm ảnh và lọc trùng (Không xoá file gốc)
scanner = ImageScanner(dataset_dir=RAW_DIR)
unique_image_paths = scanner.get_unique_filepaths()

# 2. Định dạng lại toàn bộ ảnh và lưu ra thư mục working
formatter = ImageFormatter(output_dir=CLEAN_DIR, target_size=512)
formatter.process(image_paths=unique_image_paths)

# 3. Chạy BLIP để sinh Caption
captioner = BLIPCaptioner(
    dataset_dir=CLEAN_DIR, 
    trigger_word="nhatbinh dress, traditional vietnamese clothing, "
)
captioner.process()

CLEAN_DIR = "/kaggle/working/dataset_formatted"
sanitizer = CaptionSanitizer(dataset_dir=CLEAN_DIR)
sanitizer.process()

!cat /kaggle/working/dataset_formatted/0000.txt

Scanning for images in directory: /kaggle/input/datasets/thedestoroyah/ao-nhat-binh-v2/ao_nhat_binh
Scan completed. Found 394 unique images. Ignored 74 duplicates.
Starting image formatting. Target size: 512x512
Formatting completed. Successfully processed 394 images in /kaggle/working/dataset_formatted.
Loading BLIP model on device: cuda


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Starting auto-captioning process in: /kaggle/working/dataset_formatted
Processed 50 images and generated captions...
Processed 100 images and generated captions...
Processed 150 images and generated captions...
Processed 200 images and generated captions...
Processed 250 images and generated captions...
Processed 300 images and generated captions...
Processed 350 images and generated captions...
Captioning completed. Successfully generated 394 text files.
Starting caption sanitization in directory: /kaggle/working/dataset_formatted
Sanitization completed. Cleaned 394 caption files.
nhatbinh dress, traditional vietnamese clothing, a woman in traditional traditional attire traditional attire

In [12]:
!zip -r file.zip /kaggle/working
!ls
from IPython.display import FileLink
FileLink(r'file.zip')
import zipfile
import os
from IPython.display import FileLink

def zip_dir(directory = os.curdir, file_name = 'directory.zip'):
    """
    zip all the files in a directory
    
    Parameters
    _____
    directory: str
        directory needs to be zipped, defualt is current working directory
        
    file_name: str
        the name of the zipped file (including .zip), default is 'directory.zip'
        
    Returns
    _____
    Creates a hyperlink, which can be used to download the zip file)
    """
    os.chdir(directory)
    zip_ref = zipfile.ZipFile(file_name, mode='w')
    for folder, _, files in os.walk(directory):
        for file in files:
            if file_name in file:
                pass
            else:
                zip_ref.write(os.path.join(folder, file))

    return FileLink(file_name)

zip_dir()

updating: kaggle/working/ (stored 0%)
updating: kaggle/working/dataset_formatted/ (stored 0%)
updating: kaggle/working/.virtual_documents/ (stored 0%)
updating: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 69%)
  adding: kaggle/working/directory.zip (stored 0%)
  adding: kaggle/working/dataset_formatted/0370.txt (deflated 23%)
  adding: kaggle/working/dataset_formatted/0084.txt (deflated 22%)
  adding: kaggle/working/dataset_formatted/0234.txt (deflated 38%)
  adding: kaggle/working/dataset_formatted/0118.txt (deflated 24%)
  adding: kaggle/working/dataset_formatted/0286.jpg (deflated 0%)
  adding: kaggle/working/dataset_formatted/0315.jpg (deflated 0%)
  adding: kaggle/working/dataset_formatted/0190.jpg (deflated 0%)
  adding: kaggle/working/dataset_formatted/0245.jpg (deflated 0%)
  adding: kaggle/working/dataset_formatted/0048.txt (deflated 32%)
  adding: kaggle/working/dataset_formatted/0143.jpg (deflated 0%)
  adding: kaggle/working/dataset_formatted/0381.

/kaggle/working/directory.zip